In [ ]:
# Colab setup - run this cell first
import sys, os
from pathlib import Path

if 'google.colab' in sys.modules:
    repo_dir = '/content/CompMathAndAICourse/'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/lruthotto/CompMathAndAICourse.git {repo_dir}
        %pip install -q jax jaxlib optax h5py

    NOTEBOOK_DIR = Path(repo_dir) / 'code' / '07-sciml'
    sys.path.insert(0, str(NOTEBOOK_DIR))
    print(f"Running on Colab, repo cloned to {repo_dir}")
else:
    # Local execution: get notebook directory from VS Code or fallback
    try:
        import IPython
        notebook_path = IPython.extract_module_locals()[1]['__vsc_ipynb_file__']
        NOTEBOOK_DIR = Path(notebook_path).parent
    except (KeyError, AttributeError, TypeError):
        # Fallback: search for utils.py to find the correct directory
        for candidate in [Path.cwd(), Path.cwd() / 'code' / '07-sciml']:
            if (candidate / 'utils.py').exists():
                NOTEBOOK_DIR = candidate
                break
        else:
            NOTEBOOK_DIR = Path.cwd()
    
    if str(NOTEBOOK_DIR) not in sys.path:
        sys.path.insert(0, str(NOTEBOOK_DIR))
    print(f"Running locally from {NOTEBOOK_DIR}")

# Neural Operators for Darcy Flow

**Course:** Computational Mathematics and AI  
**Lecture 7:** Scientific ML for PDEs

## Overview

This notebook demonstrates **Neural Operators** for learning solution operators of PDEs:

$$\mathcal{G}^\dagger: \kappa \mapsto u$$

where $u$ solves the Darcy flow equation:
$$-\nabla \cdot (\kappa(x,y) \nabla u) = f \quad \text{in } \Omega = [0,1]^2$$
$$u = 0 \quad \text{on } \partial\Omega$$

### Key Idea

Instead of solving **one** PDE instance (like classical solvers or PINNs), neural operators learn to map **any** input function $\kappa$ to its corresponding solution $u$.

**Benefits:**
- **Amortized cost**: Train once, solve many instances instantly
- **Generalization**: Handle varying PDE parameters without re-solving
- **GPU parallelism**: Batch inference is highly efficient

## Methods Compared

1. **FNO (Fourier Neural Operator)**: Learns in spectral domain via Fourier transforms
2. **DeepONet (Deep Operator Network)**: Branch-trunk architecture for operator learning

## Learning Objectives

1. Understand the operator learning paradigm
2. Implement and train FNO and DeepONet
3. Compare accuracy and efficiency of both approaches

In [ ]:
# =============================================================================
# Configuration
# =============================================================================
# Set TRAIN_MODE = False to skip training and load saved models for plot regeneration
TRAIN_MODE = True

# Set INTERACTIVE = True for Jupyter (shows figures inline), False for headless/saving only
INTERACTIVE = False

# Output directory for figures - set to lecture path when generating slides
# Default: NOTEBOOK_DIR / "figures" (local development)
# For slides: Set to Path("../../lectures/lecture07-scientific-ml/figures/pinn/operator")
OUTPUT_DIR = None  # Will be set after NOTEBOOK_DIR is determined

# =============================================================================
# Imports and Setup
# =============================================================================
import matplotlib
if not INTERACTIVE:
    matplotlib.use('Agg')

import jax
import jax.numpy as jnp
from jax import random, jit
import optax
import numpy as np
import matplotlib.pyplot as plt
from time import time
import pandas as pd
import pickle

from utils import (
    load_pdebench_darcy, compute_l2_error,
    plot_solution_comparison, plot_convergence
)
from models import FNO2d, create_fno, DeepONet, create_deeponet, count_params, mse_loss, rel_l2_error

# Set up JAX
print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")
print(f"Mode: {'TRAINING' if TRAIN_MODE else 'PLOT REGENERATION (loading saved models)'}")
print(f"Figures: {'INTERACTIVE' if INTERACTIVE else 'SAVING TO FILES'}")

# Master PRNG key for reproducibility
MASTER_KEY = random.PRNGKey(42)

# Directory for saved models (relative to notebook location)
MODEL_DIR = NOTEBOOK_DIR / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)

# Set output directory for figures
if OUTPUT_DIR is None:
    OUTPUT_DIR = NOTEBOOK_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Model directory: {MODEL_DIR}")


# =============================================================================
# Helper functions for saving/loading models
# =============================================================================
def save_model(params, filepath):
    """Save JAX model parameters to file."""
    with open(filepath, 'wb') as f:
        pickle.dump(params, f)
    print(f"Saved model to: {filepath}")


def load_model(filepath):
    """Load JAX model parameters from file."""
    with open(filepath, 'rb') as f:
        params = pickle.load(f)
    print(f"Loaded model from: {filepath}")
    return params


def show_or_save(fig, filepath, title=None):
    """Show figure interactively or save to file based on INTERACTIVE mode."""
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    if INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)
    print(f"Saved: {filepath}")

## Load PDEBench Data

We use the **PDEBench Darcy Flow** dataset with 10,000 pairs $(\kappa_i, u_i)$.

Unlike PINNs that solve one instance, neural operators need **training data** - pairs of inputs and outputs from the solution operator.

In [ ]:
# Data configuration
DATA_PATH = NOTEBOOK_DIR / "data" / "pdebench" / "2D_DarcyFlow_beta1.0_Train.hdf5"
INDICES_PATH = MODEL_DIR / "data_split_indices.json"

# Use larger training set for better results (PDEBench has 10,000 samples)
N_TRAIN = 8000
N_VAL = 1000
N_TEST = 1000

# Load data with train/val/test splits
# When TRAIN_MODE=True: generate new indices and save them
# When TRAIN_MODE=False: load saved indices for exact reproducibility
train_data, val_data, test_data, resolution = load_pdebench_darcy(
    DATA_PATH,
    n_train=N_TRAIN,
    n_val=N_VAL,
    n_test=N_TEST,
    seed=42,
    auto_download=True,
    normalize=False,  # PDEBench style: unnormalized
    indices_path=INDICES_PATH,
    save_indices=TRAIN_MODE,      # Save when training
    load_indices=not TRAIN_MODE,  # Load when regenerating figures
)

print(f"\nData loaded:")
print(f"  Train: {len(train_data['kappa'])} samples")
print(f"  Val:   {len(val_data['kappa'])} samples")
print(f"  Test:  {len(test_data['kappa'])} samples")
print(f"  Resolution: {resolution}x{resolution}")

In [5]:
# Visualize sample input-output pairs
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # Permeability
    im0 = axes[0, i].imshow(np.array(train_data['kappa'][i]).T, origin='lower',
                            cmap='viridis', extent=[0,1,0,1])
    axes[0, i].set_title(f'$\\kappa_{i+1}$')
    axes[0, i].set_xlabel('x')
    if i == 0:
        axes[0, i].set_ylabel('y')

    # Solution
    im1 = axes[1, i].imshow(np.array(train_data['u'][i]).T, origin='lower',
                            cmap='RdBu_r', extent=[0,1,0,1])
    axes[1, i].set_title(f'$u_{i+1}$')
    axes[1, i].set_xlabel('x')
    if i == 0:
        axes[1, i].set_ylabel('y')

plt.suptitle('PDEBench Training Samples: Permeability $\\kappa$ → Solution $u$', fontsize=14)
plt.tight_layout()
show_or_save(fig, OUTPUT_DIR / 'operator_training_samples.png')

Saved: figures-new/operator_training_samples.png


---

# Part 1: Fourier Neural Operator (FNO)

## Architecture

FNO operates in the **spectral domain** using Fourier transforms:

$$\text{FNO Block}: v \mapsto \sigma\left( W v + \mathcal{F}^{-1}(R \cdot \mathcal{F}(v))\right)$$

where:
- $\mathcal{F}$: Fast Fourier Transform
- $R$: Learnable spectral weights (truncated to low frequencies)
- $W$: Local linear transform (1x1 convolution)
- $\sigma$: Nonlinear activation

**Key insight:** By learning in Fourier space, FNO achieves **resolution invariance** - it can transfer to different grid resolutions.

In [ ]:
# Best FNO configuration from HPO (fno_pdebench_tuned.yaml)
fno_config = {
    # Architecture
    'modes': 12,        # Number of Fourier modes to keep
    'width': 20,        # Hidden channel dimension
    'n_layers': 4,      # Number of FNO blocks
    'use_grid': True,   # Append (x,y) coordinates to input

    # Training
    'learning_rate': 0.00787,
    'batch_size': 20,
    'n_epochs': 150,

    # Scheduler
    'scheduler': 'step',
    'step_size': 30,
    'gamma': 0.236,
}

print("FNO Configuration:")
for k, v in fno_config.items():
    print(f"  {k}: {v}")

In [7]:
# Initialize FNO model
key, subkey = random.split(MASTER_KEY)

fno_model = create_fno(
    modes=fno_config['modes'],
    width=fno_config['width'],
    n_layers=fno_config['n_layers'],
    use_grid=fno_config['use_grid'],
)

fno_params = fno_model.init(subkey, resolution)

n_params_fno = count_params(fno_params)
print(f"\nFNO Model:")
print(f"  Modes: {fno_config['modes']}")
print(f"  Width: {fno_config['width']}")
print(f"  Layers: {fno_config['n_layers']}")
print(f"  Parameters: {n_params_fno:,}")


FNO Model:
  Modes: 12
  Width: 20
  Layers: 4
  Parameters: 925,948


In [9]:
# Training utilities
def create_batches(data, batch_size, key):
    """Create shuffled batches from data."""
    n = len(data['kappa'])
    perm = random.permutation(key, n)
    kappa_shuffled = data['kappa'][perm]
    u_shuffled = data['u'][perm]

    n_batches = n // batch_size
    for i in range(n_batches):
        start = i * batch_size
        end = start + batch_size
        yield kappa_shuffled[start:end], u_shuffled[start:end]

def compute_val_loss(model, params, data, batch_size=50):
    """Compute validation loss and relative L2 error."""
    total_mse = 0.0
    total_rel_l2 = 0.0
    n_samples = 0

    n = len(data['kappa'])
    for i in range(0, n, batch_size):
        end = min(i + batch_size, n)
        kappa_batch = data['kappa'][i:end]
        u_batch = data['u'][i:end]

        u_pred = model.apply(params, kappa_batch)

        total_mse += float(mse_loss(u_pred, u_batch)) * len(kappa_batch)
        total_rel_l2 += rel_l2_error(u_pred, u_batch) * len(kappa_batch)
        n_samples += len(kappa_batch)

    return total_mse / n_samples, total_rel_l2 / n_samples

In [10]:
# Setup optimizer with step learning rate schedule (only needed for training)
if TRAIN_MODE:
    def create_fno_optimizer(config, steps_per_epoch):
        """Create optimizer with step decay schedule."""
        schedule = optax.exponential_decay(
            init_value=config['learning_rate'],
            transition_steps=config['step_size'] * steps_per_epoch,
            decay_rate=config['gamma'],
        )
        return optax.adam(schedule)

    steps_per_epoch = N_TRAIN // fno_config['batch_size']
    fno_optimizer = create_fno_optimizer(fno_config, steps_per_epoch)
    fno_opt_state = fno_optimizer.init(fno_params)

    # JIT-compiled training step
    @jit
    def fno_train_step(params, opt_state, kappa_batch, u_batch):
        def loss_fn(params):
            u_pred = fno_model.apply(params, kappa_batch)
            return mse_loss(u_pred, u_batch)

        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = fno_optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    print(f"Training setup complete")
    print(f"  Steps per epoch: {steps_per_epoch}")
    print(f"  Total steps: {steps_per_epoch * fno_config['n_epochs']}")
else:
    print("Skipping FNO training setup (TRAIN_MODE=False)")

Training setup complete
  Steps per epoch: 400
  Total steps: 60000


In [11]:
# Training loop (or load saved model)
FNO_MODEL_PATH = MODEL_DIR / 'best_fno_params.pkl'

if TRAIN_MODE:
    print("\nTraining FNO...")
    print("-" * 70)

    fno_history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_rel_l2': []}
    log_every = 10

    key = random.PRNGKey(0)
    start_time = time()
    best_val_loss = float('inf')
    best_fno_params = fno_params

    for epoch in range(fno_config['n_epochs']):
        key, subkey = random.split(key)

        # Train epoch
        epoch_loss = 0.0
        n_batches = 0

        for kappa_batch, u_batch in create_batches(train_data, fno_config['batch_size'], subkey):
            fno_params, fno_opt_state, loss = fno_train_step(fno_params, fno_opt_state, kappa_batch, u_batch)
            epoch_loss += float(loss)
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches

        # Log progress
        if epoch % log_every == 0 or epoch == fno_config['n_epochs'] - 1:
            val_loss, val_rel_l2 = compute_val_loss(fno_model, fno_params, val_data)
            elapsed = time() - start_time

            fno_history['epoch'].append(epoch)
            fno_history['train_loss'].append(avg_train_loss)
            fno_history['val_loss'].append(val_loss)
            fno_history['val_rel_l2'].append(val_rel_l2)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_fno_params = fno_params

            print(f"Epoch {epoch:3d} | Train: {avg_train_loss:.6e} | Val: {val_loss:.6e} | "
                  f"Val L2: {val_rel_l2:.4f} ({val_rel_l2*100:.1f}%) | Time: {elapsed:.1f}s")

    fno_training_time = time() - start_time
    print("-" * 70)
    print(f"FNO training complete in {fno_training_time:.1f}s")

    # Save best model
    save_model(best_fno_params, FNO_MODEL_PATH)

else:
    # Load saved model
    if FNO_MODEL_PATH.exists():
        best_fno_params = load_model(FNO_MODEL_PATH)
        # Load training history if available
        fno_history_path = OUTPUT_DIR / 'fno_training_history.csv'
        if fno_history_path.exists():
            fno_df = pd.read_csv(fno_history_path)
            fno_history = fno_df.to_dict('list')
            print(f"Loaded training history from: {fno_history_path}")
        else:
            fno_history = None
        fno_training_time = None
    else:
        raise FileNotFoundError(f"No saved model found at {FNO_MODEL_PATH}. Run with TRAIN_MODE=True first.")


Training FNO...
----------------------------------------------------------------------
Epoch   0 | Train: 2.359456e-03 | Val: 6.991724e-04 | Val L2: 0.1208 (12.1%) | Time: 13.7s
Epoch  10 | Train: 2.491233e-04 | Val: 4.089586e-04 | Val L2: 0.0907 (9.1%) | Time: 62.2s
Epoch  20 | Train: 1.117757e-04 | Val: 3.377079e-04 | Val L2: 0.0812 (8.1%) | Time: 110.4s
Epoch  30 | Train: 7.530730e-05 | Val: 3.199937e-04 | Val L2: 0.0782 (7.8%) | Time: 158.8s
Epoch  40 | Train: 5.941493e-05 | Val: 2.984090e-04 | Val L2: 0.0752 (7.5%) | Time: 207.2s
Epoch  50 | Train: 4.897499e-05 | Val: 3.022611e-04 | Val L2: 0.0752 (7.5%) | Time: 255.5s
Epoch  60 | Train: 4.367707e-05 | Val: 2.882460e-04 | Val L2: 0.0733 (7.3%) | Time: 303.7s
Epoch  70 | Train: 4.272611e-05 | Val: 2.904004e-04 | Val L2: 0.0737 (7.4%) | Time: 352.2s
Epoch  80 | Train: 4.034177e-05 | Val: 2.888855e-04 | Val L2: 0.0735 (7.3%) | Time: 401.0s
Epoch  90 | Train: 3.918578e-05 | Val: 2.865099e-04 | Val L2: 0.0731 (7.3%) | Time: 449.5s
Epo

In [12]:
# Save FNO training history (only in training mode)
if TRAIN_MODE and fno_history:
    fno_df = pd.DataFrame(fno_history)
    fno_df.to_csv(OUTPUT_DIR / 'fno_training_history.csv', index=False)
    print(f"Saved: {OUTPUT_DIR / 'fno_training_history.csv'}")
    print(fno_df.tail())
elif fno_history:
    print("Training history loaded from file:")
    print(pd.DataFrame(fno_history).tail())

Saved: figures-new/fno_training_history.csv
    epoch  train_loss  val_loss  val_rel_l2
11    110    0.000039  0.000284    0.072820
12    120    0.000038  0.000285    0.072928
13    130    0.000038  0.000284    0.072842
14    140    0.000038  0.000284    0.072819
15    149    0.000038  0.000284    0.072838


In [ ]:
# Evaluate FNO on test set
test_loss_fno, test_rel_l2_fno = compute_val_loss(fno_model, best_fno_params, test_data)

print(f"\nFNO Test Results:")
print(f"  Test MSE Loss: {test_loss_fno:.6e}")
print(f"  Test Relative L2: {test_rel_l2_fno:.4f} ({test_rel_l2_fno*100:.1f}%)")

# Always compute inference time (needed for comparison table)
start = time()
for _ in range(10):
    _ = fno_model.apply(best_fno_params, test_data['kappa'])
fno_inference_time = (time() - start) / 10 / len(test_data['kappa']) * 1000
print(f"  Inference time: {fno_inference_time:.2f} ms/sample")

In [14]:
# Visualize FNO predictions
test_idx = 0
u_fno_pred = fno_model.apply(best_fno_params, test_data['kappa'][test_idx:test_idx+1])[0]

fig = plot_solution_comparison(
    np.array(test_data['kappa'][test_idx]),
    np.array(test_data['u'][test_idx]),
    np.array(u_fno_pred),
    title='FNO Prediction vs Reference'
)
show_or_save(fig, OUTPUT_DIR / 'fno_prediction_sample0.png')

Saved: figures-new/fno_prediction_sample0.png


---

# Part 2: Deep Operator Network (DeepONet)

## Architecture

DeepONet uses a **branch-trunk** architecture:

$$\mathcal{G}_\theta(\kappa)(x,y) = \sum_{k=1}^p b_k(\kappa) \cdot t_k(x, y)$$

where:
- **Branch network** $b_k$: Encodes the input function $\kappa$ at sensor locations
- **Trunk network** $t_k$: Encodes the query coordinates $(x, y)$
- Output: Inner product of branch and trunk embeddings

**Key insight:** By separating function encoding from coordinate encoding, DeepONet can query the solution at arbitrary points.

In [15]:
# DeepONet configuration (deeponet_slides.yaml - best results)
deeponet_config = {
    # Architecture
    'n_sensors': 4096,          # 64x64 sensor grid (samples from 128x128)
    'latent_dim': 256,          # Dimension of branch/trunk embedding
    'branch_hidden': [512, 512, 512],  # Branch network hidden layers
    'trunk_hidden': [512, 512, 512],   # Trunk network hidden layers

    # Training
    'learning_rate': 0.0001,
    'weight_decay': 0.0001,     # L2 regularization (important for DeepONet!)
    'batch_size': 16,
    'n_epochs': 300,

    # Scheduler
    'scheduler': 'cosine',
}

print("DeepONet Configuration:")
for k, v in deeponet_config.items():
    print(f"  {k}: {v}")

DeepONet Configuration:
  n_sensors: 4096
  latent_dim: 256
  branch_hidden: [512, 512, 512]
  trunk_hidden: [512, 512, 512]
  learning_rate: 0.0001
  weight_decay: 0.0001
  batch_size: 16
  n_epochs: 300
  scheduler: cosine


In [16]:
# Initialize DeepONet model
# Use a fresh key (independent of FNO training)
deeponet_key = random.PRNGKey(100)
_, subkey = random.split(deeponet_key)

deeponet_model = create_deeponet(
    n_sensors=deeponet_config['n_sensors'],
    latent_dim=deeponet_config['latent_dim'],
    branch_hidden=deeponet_config['branch_hidden'],
    trunk_hidden=deeponet_config['trunk_hidden'],
)

deeponet_params = deeponet_model.init(subkey, resolution)

n_params_deeponet = count_params(deeponet_params)
print(f"\nDeepONet Model:")
print(f"  Sensors: {deeponet_config['n_sensors']}")
print(f"  Latent dim: {deeponet_config['latent_dim']}")
print(f"  Branch hidden: {deeponet_config['branch_hidden']}")
print(f"  Trunk hidden: {deeponet_config['trunk_hidden']}")
print(f"  Parameters: {n_params_deeponet:,}")


DeepONet Model:
  Sensors: 4096
  Latent dim: 256
  Branch hidden: [512, 512, 512]
  Trunk hidden: [512, 512, 512]
  Parameters: 3,453,440


In [17]:
# Setup optimizer with cosine annealing schedule and weight decay (only needed for training)
if TRAIN_MODE:
    def create_deeponet_optimizer(config, total_steps):
        """Create AdamW optimizer with cosine annealing schedule."""
        schedule = optax.cosine_decay_schedule(
            init_value=config['learning_rate'],
            decay_steps=total_steps,
        )
        # Use AdamW for weight decay regularization
        return optax.adamw(schedule, weight_decay=config['weight_decay'])

    steps_per_epoch_don = N_TRAIN // deeponet_config['batch_size']
    total_steps_don = steps_per_epoch_don * deeponet_config['n_epochs']

    deeponet_optimizer = create_deeponet_optimizer(deeponet_config, total_steps_don)
    deeponet_opt_state = deeponet_optimizer.init(deeponet_params)

    # JIT-compiled training step
    @jit
    def deeponet_train_step(params, opt_state, kappa_batch, u_batch):
        def loss_fn(params):
            u_pred = deeponet_model.apply(params, kappa_batch)
            return mse_loss(u_pred, u_batch)

        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = deeponet_optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    print(f"Training setup complete")
    print(f"  Optimizer: AdamW (weight_decay={deeponet_config['weight_decay']})")
    print(f"  Steps per epoch: {steps_per_epoch_don}")
    print(f"  Total steps: {total_steps_don}")
else:
    print("Skipping DeepONet training setup (TRAIN_MODE=False)")

Training setup complete
  Optimizer: AdamW (weight_decay=0.0001)
  Steps per epoch: 500
  Total steps: 150000


In [18]:
# Training loop (or load saved model)
DEEPONET_MODEL_PATH = MODEL_DIR / 'best_deeponet_params.pkl'

if TRAIN_MODE:
    print("\nTraining DeepONet...")
    print("-" * 70)

    deeponet_history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_rel_l2': []}
    log_every = 20

    key = random.PRNGKey(1)
    start_time = time()
    best_val_loss_don = float('inf')
    best_deeponet_params = deeponet_params

    for epoch in range(deeponet_config['n_epochs']):
        key, subkey = random.split(key)

        # Train epoch
        epoch_loss = 0.0
        n_batches = 0

        for kappa_batch, u_batch in create_batches(train_data, deeponet_config['batch_size'], subkey):
            deeponet_params, deeponet_opt_state, loss = deeponet_train_step(
                deeponet_params, deeponet_opt_state, kappa_batch, u_batch
            )
            epoch_loss += float(loss)
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches

        # Log progress
        if epoch % log_every == 0 or epoch == deeponet_config['n_epochs'] - 1:
            val_loss, val_rel_l2 = compute_val_loss(deeponet_model, deeponet_params, val_data)
            elapsed = time() - start_time

            deeponet_history['epoch'].append(epoch)
            deeponet_history['train_loss'].append(avg_train_loss)
            deeponet_history['val_loss'].append(val_loss)
            deeponet_history['val_rel_l2'].append(val_rel_l2)

            if val_loss < best_val_loss_don:
                best_val_loss_don = val_loss
                best_deeponet_params = deeponet_params

            print(f"Epoch {epoch:3d} | Train: {avg_train_loss:.6e} | Val: {val_loss:.6e} | "
                  f"Val L2: {val_rel_l2:.4f} ({val_rel_l2*100:.1f}%) | Time: {elapsed:.1f}s")

    deeponet_training_time = time() - start_time
    print("-" * 70)
    print(f"DeepONet training complete in {deeponet_training_time:.1f}s")

    # Save best model
    save_model(best_deeponet_params, DEEPONET_MODEL_PATH)

else:
    # Load saved model
    if DEEPONET_MODEL_PATH.exists():
        best_deeponet_params = load_model(DEEPONET_MODEL_PATH)
        # Load training history if available
        deeponet_history_path = OUTPUT_DIR / 'deeponet_training_history.csv'
        if deeponet_history_path.exists():
            deeponet_df = pd.read_csv(deeponet_history_path)
            deeponet_history = deeponet_df.to_dict('list')
            print(f"Loaded training history from: {deeponet_history_path}")
        else:
            deeponet_history = None
        deeponet_training_time = None
    else:
        raise FileNotFoundError(f"No saved model found at {DEEPONET_MODEL_PATH}. Run with TRAIN_MODE=True first.")


Training DeepONet...
----------------------------------------------------------------------
Epoch   0 | Train: 9.703480e-03 | Val: 5.419902e-03 | Val L2: 0.3385 (33.8%) | Time: 13.8s
Epoch  20 | Train: 1.539330e-03 | Val: 1.668207e-03 | Val L2: 0.1890 (18.9%) | Time: 58.3s
Epoch  40 | Train: 6.825831e-04 | Val: 9.065173e-04 | Val L2: 0.1376 (13.8%) | Time: 103.6s
Epoch  60 | Train: 3.901847e-04 | Val: 6.237506e-04 | Val L2: 0.1128 (11.3%) | Time: 147.7s
Epoch  80 | Train: 2.891371e-04 | Val: 5.852543e-04 | Val L2: 0.1083 (10.8%) | Time: 192.1s
Epoch 100 | Train: 2.451333e-04 | Val: 5.491907e-04 | Val L2: 0.1047 (10.5%) | Time: 238.0s
Epoch 120 | Train: 1.990190e-04 | Val: 4.792743e-04 | Val L2: 0.0970 (9.7%) | Time: 283.0s
Epoch 140 | Train: 1.604898e-04 | Val: 4.444714e-04 | Val L2: 0.0936 (9.4%) | Time: 328.1s
Epoch 160 | Train: 1.342819e-04 | Val: 4.177508e-04 | Val L2: 0.0899 (9.0%) | Time: 374.5s
Epoch 180 | Train: 1.260678e-04 | Val: 4.027800e-04 | Val L2: 0.0878 (8.8%) | Time: 

In [19]:
# Save DeepONet training history (only in training mode)
if TRAIN_MODE and deeponet_history:
    deeponet_df = pd.DataFrame(deeponet_history)
    deeponet_df.to_csv(OUTPUT_DIR / 'deeponet_training_history.csv', index=False)
    print(f"Saved: {OUTPUT_DIR / 'deeponet_training_history.csv'}")
    print(deeponet_df.tail())
elif deeponet_history:
    print("Training history loaded from file:")
    print(pd.DataFrame(deeponet_history).tail())

Saved: figures-new/deeponet_training_history.csv
    epoch  train_loss  val_loss  val_rel_l2
11    220    0.000106  0.000386    0.085871
12    240    0.000099  0.000379    0.085157
13    260    0.000096  0.000382    0.085187
14    280    0.000094  0.000380    0.084910
15    299    0.000093  0.000380    0.084945


In [ ]:
# Evaluate DeepONet on test set
test_loss_don, test_rel_l2_don = compute_val_loss(deeponet_model, best_deeponet_params, test_data)

print(f"\nDeepONet Test Results:")
print(f"  Test MSE Loss: {test_loss_don:.6e}")
print(f"  Test Relative L2: {test_rel_l2_don:.4f} ({test_rel_l2_don*100:.1f}%)")

# Always compute inference time (needed for comparison table)
start = time()
for _ in range(10):
    _ = deeponet_model.apply(best_deeponet_params, test_data['kappa'])
deeponet_inference_time = (time() - start) / 10 / len(test_data['kappa']) * 1000
print(f"  Inference time: {deeponet_inference_time:.2f} ms/sample")

In [21]:
# Visualize DeepONet predictions
u_don_pred = deeponet_model.apply(best_deeponet_params, test_data['kappa'][test_idx:test_idx+1])[0]

fig = plot_solution_comparison(
    np.array(test_data['kappa'][test_idx]),
    np.array(test_data['u'][test_idx]),
    np.array(u_don_pred),
    title='DeepONet Prediction vs Reference'
)
show_or_save(fig, OUTPUT_DIR / 'deeponet_prediction_sample0.png')

Saved: figures-new/deeponet_prediction_sample0.png


---

# Part 3: Comparison

Let's compare FNO and DeepONet across multiple metrics.

In [22]:
# Compare training convergence (only if history is available)
if fno_history and deeponet_history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    axes[0].semilogy(fno_history['epoch'], fno_history['train_loss'], 'b-', linewidth=2, label='FNO Train')
    axes[0].semilogy(fno_history['epoch'], fno_history['val_loss'], 'b--', linewidth=2, label='FNO Val')
    axes[0].semilogy(deeponet_history['epoch'], deeponet_history['train_loss'], 'r-', linewidth=2, label='DeepONet Train')
    axes[0].semilogy(deeponet_history['epoch'], deeponet_history['val_loss'], 'r--', linewidth=2, label='DeepONet Val')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('MSE Loss', fontsize=12)
    axes[0].set_title('Training Convergence', fontsize=14)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # Relative L2 error
    axes[1].plot(fno_history['epoch'], [x*100 for x in fno_history['val_rel_l2']], 'b-', linewidth=2, label='FNO')
    axes[1].plot(deeponet_history['epoch'], [x*100 for x in deeponet_history['val_rel_l2']], 'r-', linewidth=2, label='DeepONet')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Relative L2 Error (%)', fontsize=12)
    axes[1].set_title('Validation Error', fontsize=14)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    show_or_save(fig, OUTPUT_DIR / 'operator_training_comparison.png')
else:
    print("Training history not available - skipping convergence plot")

Saved: figures-new/operator_training_comparison.png


In [ ]:
# Summary comparison table
print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Metric':<25} {'FNO':>20} {'DeepONet':>20}")
print("-" * 70)
print(f"{'Parameters':<25} {n_params_fno:>20,} {n_params_deeponet:>20,}")
print(f"{'Test Rel. L2 Error':<25} {test_rel_l2_fno*100:>19.1f}% {test_rel_l2_don*100:>19.1f}%")

# Handle training time - may be None when models are loaded
if TRAIN_MODE and fno_training_time is not None and deeponet_training_time is not None:
    print(f"{'Training Time':<25} {fno_training_time:>18.1f}s {deeponet_training_time:>18.1f}s")
else:
    print(f"{'Training Time':<25} {'(loaded)':>20} {'(loaded)':>20}")

print(f"{'Inference Time':<25} {fno_inference_time:>17.2f}ms {deeponet_inference_time:>17.2f}ms")
print(f"{'Epochs':<25} {fno_config['n_epochs']:>20} {deeponet_config['n_epochs']:>20}")
print("=" * 70)

In [ ]:
# Save 3 test examples with predictions from both models
# Using new naming convention: method_component.png
test_indices = [0, 1, 2]

for idx in test_indices:
    kappa = np.array(test_data['kappa'][idx])
    u_ref = np.array(test_data['u'][idx])
    u_fno = np.array(fno_model.apply(best_fno_params, test_data['kappa'][idx:idx+1])[0])
    u_don = np.array(deeponet_model.apply(best_deeponet_params, test_data['kappa'][idx:idx+1])[0])

    err_fno = compute_l2_error(u_fno, u_ref)
    err_don = compute_l2_error(u_don, u_ref)

    # Get consistent colorbar limits for u plots
    u_vmin = min(u_ref.min(), u_fno.min(), u_don.min())
    u_vmax = max(u_ref.max(), u_fno.max(), u_don.max())
    err_max = max(np.abs(u_fno - u_ref).max(), np.abs(u_don - u_ref).max())

    # === Save individual figures with new naming convention ===
    
    # Shared: kappa and reference (same for both methods)
    save_individual_figure(kappa, f'$\\kappa$ (test {idx})',
                          OUTPUT_DIR / f'operator_test{idx}_kappa.png', cmap='viridis')
    save_individual_figure(u_ref, f'Reference $u$ (test {idx})',
                          OUTPUT_DIR / f'operator_test{idx}_reference.png', cmap='RdBu_r',
                          vmin=u_vmin, vmax=u_vmax)
    
    # FNO
    save_individual_figure(u_fno, f'FNO ({err_fno*100:.1f}%)',
                          OUTPUT_DIR / f'fno_test{idx}_prediction.png', cmap='RdBu_r',
                          vmin=u_vmin, vmax=u_vmax)
    save_individual_figure(np.abs(u_fno - u_ref), '|FNO Error|',
                          OUTPUT_DIR / f'fno_test{idx}_error.png', cmap='hot',
                          vmin=0, vmax=err_max)
    
    # DeepONet
    save_individual_figure(u_don, f'DeepONet ({err_don*100:.1f}%)',
                          OUTPUT_DIR / f'deeponet_test{idx}_prediction.png', cmap='RdBu_r',
                          vmin=u_vmin, vmax=u_vmax)
    save_individual_figure(np.abs(u_don - u_ref), '|DeepONet Error|',
                          OUTPUT_DIR / f'deeponet_test{idx}_error.png', cmap='hot',
                          vmin=0, vmax=err_max)

    print(f"Test {idx}: FNO error={err_fno*100:.1f}%, DeepONet error={err_don*100:.1f}%")

In [ ]:
# Save comparison summary to CSV
comparison_df = pd.DataFrame({
    'metric': ['Parameters', 'Test_Rel_L2_Error', 'Training_Time_s', 'Inference_Time_ms', 'Epochs'],
    'FNO': [n_params_fno, test_rel_l2_fno, fno_training_time if TRAIN_MODE else 'loaded', fno_inference_time, fno_config['n_epochs']],
    'DeepONet': [n_params_deeponet, test_rel_l2_don, deeponet_training_time if TRAIN_MODE else 'loaded', deeponet_inference_time, deeponet_config['n_epochs']]
})
comparison_df.to_csv(OUTPUT_DIR / 'operator_comparison_summary.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'operator_comparison_summary.csv'}")
print(comparison_df)

---

# Part 4: Spectral Bias Analysis

Neural operators also exhibit **spectral bias** - but it manifests differently than in PINNs.

Key questions:
1. Do FNO and DeepONet learn different frequency components?
2. How does the error distribute across frequencies?
3. Does FNO's Fourier-space learning help with high frequencies?

In [ ]:
# Spectral analysis on a test sample
sample_idx = 0
u_ref = np.array(test_data['u'][sample_idx])
u_fno = np.array(fno_model.apply(best_fno_params, test_data['kappa'][sample_idx:sample_idx+1])[0])
u_don = np.array(deeponet_model.apply(best_deeponet_params, test_data['kappa'][sample_idx:sample_idx+1])[0])

# Compute FFTs
ref_fft = np.abs(np.fft.fftshift(np.fft.fft2(u_ref)))
fno_fft = np.abs(np.fft.fftshift(np.fft.fft2(u_fno)))
don_fft = np.abs(np.fft.fftshift(np.fft.fft2(u_don)))

fno_error_fft = np.abs(np.fft.fftshift(np.fft.fft2(u_fno - u_ref)))
don_error_fft = np.abs(np.fft.fftshift(np.fft.fft2(u_don - u_ref)))

# Plot spectra
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Top row: solution spectra
vmax = np.log10(max(ref_fft.max(), fno_fft.max(), don_fft.max()) + 1e-10)
vmin = vmax - 6

im0 = axes[0, 0].imshow(np.log10(ref_fft + 1e-10), cmap='viridis', vmin=vmin, vmax=vmax)
axes[0, 0].set_title('Reference Spectrum')
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(np.log10(fno_fft + 1e-10), cmap='viridis', vmin=vmin, vmax=vmax)
axes[0, 1].set_title('FNO Spectrum')
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[0, 2].imshow(np.log10(don_fft + 1e-10), cmap='viridis', vmin=vmin, vmax=vmax)
axes[0, 2].set_title('DeepONet Spectrum')
plt.colorbar(im2, ax=axes[0, 2])

# Bottom row: error spectra
err_vmax = np.log10(max(fno_error_fft.max(), don_error_fft.max()) + 1e-10)
err_vmin = err_vmax - 6

im3 = axes[1, 0].imshow(np.log10(fno_error_fft + 1e-10), cmap='hot', vmin=err_vmin, vmax=err_vmax)
axes[1, 0].set_title('FNO Error Spectrum')
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(np.log10(don_error_fft + 1e-10), cmap='hot', vmin=err_vmin, vmax=err_vmax)
axes[1, 1].set_title('DeepONet Error Spectrum')
plt.colorbar(im4, ax=axes[1, 1])

# Difference in error spectra
diff = np.log10(don_error_fft + 1e-10) - np.log10(fno_error_fft + 1e-10)
im5 = axes[1, 2].imshow(diff, cmap='RdBu_r', vmin=-2, vmax=2)
axes[1, 2].set_title('log(DeepONet err) - log(FNO err)')
plt.colorbar(im5, ax=axes[1, 2])

plt.suptitle('Spectral Analysis: FNO vs DeepONet', fontsize=14)
plt.tight_layout()
show_or_save(fig, OUTPUT_DIR / 'operator_spectral_analysis.png')

In [ ]:
# Radially averaged spectrum analysis
def radial_profile(data):
    """Compute radially averaged profile of 2D data."""
    center = np.array(data.shape) // 2
    y, x = np.ogrid[:data.shape[0], :data.shape[1]]
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)

    r_max = min(center)
    tbin = np.bincount(r.ravel(), data.ravel())
    nr = np.bincount(r.ravel())
    radialprofile = tbin / (nr + 1e-10)
    return radialprofile[:r_max]

# Compute radial profiles
r_ref = radial_profile(ref_fft)
r_fno = radial_profile(fno_fft)
r_don = radial_profile(don_fft)
r_fno_err = radial_profile(fno_error_fft)
r_don_err = radial_profile(don_error_fft)

freqs = np.arange(len(r_ref))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Power spectra comparison
axes[0].semilogy(freqs, r_ref, 'k-', linewidth=2, label='Reference')
axes[0].semilogy(freqs, r_fno, 'b--', linewidth=2, label='FNO')
axes[0].semilogy(freqs, r_don, 'r--', linewidth=2, label='DeepONet')
axes[0].set_xlabel('Frequency (radial)', fontsize=12)
axes[0].set_ylabel('Power', fontsize=12)
axes[0].set_title('Radially Averaged Power Spectrum', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Error by frequency
axes[1].semilogy(freqs, r_fno_err, 'b-', linewidth=2, label='FNO Error')
axes[1].semilogy(freqs, r_don_err, 'r-', linewidth=2, label='DeepONet Error')
axes[1].axvline(x=fno_config['modes'], color='b', linestyle=':', alpha=0.7, label=f'FNO modes={fno_config["modes"]}')
axes[1].set_xlabel('Frequency (radial)', fontsize=12)
axes[1].set_ylabel('Error Power', fontsize=12)
axes[1].set_title('Error Distribution by Frequency', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
show_or_save(fig, OUTPUT_DIR / 'operator_spectral_bias.png')

# Save spectral data
spectral_df = pd.DataFrame({
    'frequency': freqs,
    'reference_power': r_ref,
    'fno_power': r_fno,
    'deeponet_power': r_don,
    'fno_error_power': r_fno_err,
    'deeponet_error_power': r_don_err
})
spectral_df.to_csv(OUTPUT_DIR / 'operator_spectral_data.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'operator_spectral_data.csv'}")

In [ ]:
# Quantify spectral bias
low_freq_cutoff = len(freqs) // 4  # First 25% of frequencies
mid_freq_cutoff = len(freqs) // 2  # First 50% of frequencies

def freq_error_breakdown(err_profile):
    total = np.sum(err_profile)
    low = np.sum(err_profile[:low_freq_cutoff])
    mid = np.sum(err_profile[low_freq_cutoff:mid_freq_cutoff])
    high = np.sum(err_profile[mid_freq_cutoff:])
    return low/total*100, mid/total*100, high/total*100

fno_low, fno_mid, fno_high = freq_error_breakdown(r_fno_err)
don_low, don_mid, don_high = freq_error_breakdown(r_don_err)

print("\n" + "=" * 60)
print("SPECTRAL ERROR DISTRIBUTION")
print("=" * 60)
print(f"{'Frequency Band':<20} {'FNO':>15} {'DeepONet':>15}")
print("-" * 60)
print(f"{'Low (0-25%)':<20} {fno_low:>14.1f}% {don_low:>14.1f}%")
print(f"{'Mid (25-50%)':<20} {fno_mid:>14.1f}% {don_mid:>14.1f}%")
print(f"{'High (50-100%)':<20} {fno_high:>14.1f}% {don_high:>14.1f}%")
print("=" * 60)
print(f"\nNote: FNO keeps {fno_config['modes']} Fourier modes, which corresponds to")
print(f"frequency cutoff at ~{fno_config['modes']/len(freqs)*100:.0f}% of Nyquist.")

## Summary

### Key Takeaways

1. **Neural operators learn solution mappings**, not individual solutions
2. **FNO** uses spectral learning (Fourier space) - explicitly controls frequency content
3. **DeepONet** separates function and coordinate encoding - flexible but has spectral bias
4. **Both achieve ~8-11% error** on PDEBench Darcy Flow

### Spectral Bias Comparison

- **FNO**: Error truncated at mode cutoff (by design). Good low-frequency recovery.
- **DeepONet**: Error distributed across all frequencies. More high-frequency error.
 
### References

1. **FNO**: Li et al. "Fourier Neural Operator for Parametric Partial Differential Equations" (2021)
2. **DeepONet**: Lu et al. "Learning Nonlinear Operators via DeepONet" (2021)
3. **PDEBench**: Takamoto et al. "PDEBench: An Extensive Benchmark for Scientific Machine Learning" (2022)

In [ ]:
# Final summary of saved files
print("\n" + "=" * 60)
print("SAVED FILES")
print("=" * 60)
print("\n--- Figures ---")
for f in sorted(OUTPUT_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} ({size_kb:.1f} KB)")
print("\n--- Models ---")
for f in sorted(MODEL_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} ({size_kb:.1f} KB)")
print("=" * 60)
print(f"\nTo regenerate plots without retraining:")
print(f"  1. Set TRAIN_MODE = False in the first cell")
print(f"  2. Run all cells")